# 2.5 Data Validation — Great Expectations & Pydantic> Automated data quality checks for your pipelines.---

## Pydantic — Schema Validation☕ **Java parallel:** Like Bean Validation (`@NotNull`, `@Size`) + Jackson deserialization.

In [ ]:
from pydantic import BaseModel, Field, field_validatorfrom datetime import datefrom enum import Enumclass AdmissionType(str, Enum):    INPATIENT = "inpatient"    OUTPATIENT = "outpatient"    EMERGENCY = "emergency"class PatientRecord(BaseModel):    patient_id: str = Field(..., pattern=r"^P\d{3,}$")    name: str = Field(..., min_length=1, max_length=100)    age: int = Field(..., ge=0, le=150)    admission_type: AdmissionType    admission_date: date    cost: float = Field(..., gt=0)    diagnosis_codes: list[str] = Field(default_factory=list)    @field_validator("diagnosis_codes")    @classmethod    def validate_icd_codes(cls, codes):        for code in codes:            if not code[0].isalpha():                raise ValueError(f"ICD code must start with a letter: {code}")        return codes# Valid recordpatient = PatientRecord(    patient_id="P001",    name="John Doe",    age=45,    admission_type="inpatient",    admission_date="2024-01-15",    cost=1200.50,    diagnosis_codes=["E11.9", "I10"],)print(patient.model_dump_json(indent=2))

In [ ]:
# Invalid records — Pydantic catches the errorsfrom pydantic import ValidationErrorinvalid_cases = [    {"patient_id": "invalid", "name": "X", "age": 45, "admission_type": "inpatient",     "admission_date": "2024-01-01", "cost": 100},    {"patient_id": "P001", "name": "Y", "age": -5, "admission_type": "inpatient",     "admission_date": "2024-01-01", "cost": 100},]for i, case in enumerate(invalid_cases):    try:        PatientRecord(**case)    except ValidationError as e:        print(f"Case {i+1} errors:")        for err in e.errors():            print(f"  {err['loc'][0]}: {err['msg']}")

## Batch Validation PatternUseful for validating rows in a data pipeline before loading.

In [ ]:
import pandas as pdfrom pydantic import ValidationErrordef validate_batch(df: pd.DataFrame, model: type[BaseModel]) -> tuple[list, list]:    """Validate each row against a Pydantic model.    Returns (valid_records, error_records)."""    valid, errors = [], []    for idx, row in df.iterrows():        try:            record = model(**row.to_dict())            valid.append(record.model_dump())        except ValidationError as e:            errors.append({"row": idx, "errors": e.errors()})    return valid, errors# Exampleraw_df = pd.DataFrame({    "patient_id": ["P001", "bad_id", "P003"],    "name": ["Alice", "Bob", ""],    "age": [30, 45, 200],    "admission_type": ["inpatient", "outpatient", "inpatient"],    "admission_date": ["2024-01-01", "2024-02-01", "2024-03-01"],    "cost": [1000, 2000, 500],})valid, errors = validate_batch(raw_df, PatientRecord)print(f"Valid: {len(valid)}, Errors: {len(errors)}")for e in errors:    print(f"  Row {e['row']}: {[err['loc'] for err in e['errors']]}")